<a href="https://colab.research.google.com/github/ibnu-ahmedin/afaan-oromo-hybrid-sentiment-analysis/blob/main/XLM_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# XLM-R MODEL
# ==========================================================

!pip install -U transformers datasets accelerate scikit-learn -q

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

# =====================================
# 1. REPRODUCIBILITY
# =====================================

torch.manual_seed(42)
np.random.seed(42)

# =====================================
# 2. LOAD DATA SPLITS
# =====================================

train_df = pd.read_csv('/content/train.csv')
val_df   = pd.read_csv('/content/val.csv')
test_df  = pd.read_csv('/content/test.csv')

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)   # MUST match (e.g., 1001)

# =====================================
# 3. LABEL MAPPING
# =====================================

label_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

# Ensure label column exists
for df in [train_df, val_df, test_df]:
    if "label" not in df.columns:
        df["label"] = df["Sentiment"].map(label_map)

# =====================================
# 4. CONVERT TO HF DATASET
# =====================================

train_dataset = Dataset.from_pandas(train_df[["Text", "label"]])
val_dataset   = Dataset.from_pandas(val_df[["Text", "label"]])
test_dataset  = Dataset.from_pandas(test_df[["Text", "label"]])

# =====================================
# 5. LOAD TOKENIZER
# =====================================

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_function(example):
    return tokenizer(
        example["Text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =====================================
# 6. LOAD MODEL
# =====================================

model = XLMRobertaForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3,
    id2label=id2label,
    label2id=label_map
)

# =====================================
# 7. METRICS
# =====================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

# =====================================
# 8. TRAINING ARGUMENTS
# =====================================

training_args = TrainingArguments(
    output_dir="./xlmr_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.02,
    warmup_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=200,
    save_total_limit=2,
    seed=42
)

# =====================================
# 9. TRAINER
# =====================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# =====================================
# 10. TRAIN
# =====================================

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

trainer.train()

# =====================================
# 11. TEST EVALUATION (SAME TEST SET)
# =====================================

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

accuracy = accuracy_score(test_df["label"], preds)

print("\n==============================")
print("XLM-R RESULTS (ALIGNED)")
print("==============================")
print("Test Size:", len(test_df))
print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:\n")
print(classification_report(
    test_df["label"],
    preds,
    target_names=["Negative", "Neutral", "Positive"]
))

# =====================================
# 12. CONFUSION MATRIX
# =====================================

cm = confusion_matrix(test_df["label"], preds)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Neutral", "Positive"]
).plot(cmap="Blues")

plt.title("XLM-R Confusion Matrix")
plt.show()

pd.DataFrame({
    "y_true": test_df["label"].values,
    "afri_preds": preds
}).to_csv("/content/xlmr_preds.csv", index=False)

print("\nPredictions saved for statistical testing")